# Energy Building Modelling

**Lab Report — Smart Cities (4EUS4HAT)**  
École nationale supérieure de l'énergie, l'eau et l'environnement

*Tran Lam The Thinh · Muth Kanhaputheavy · Hosseini Romina · Montagne Édouard*

---

# 1. Introduction

First and foremost, we have modeled the room we will focus on. We considered a 30 $m^2$ apartment with one window and one door. The room is situated in the middle of the building, so that the floor, ceiling and outdoor walls are not in contact with the exterior. The floor is covered with floorboard, the window is made with double glazing glass and the door is made of wood. The walls are 3m tall and are insulated.

The diagram representing our room is shown below:

<!-- Figure: Building_Plan.png — Room plan -->

![Building Plan](Building_Plan.png "Title")

## 1.1 General Information

- **Location**
  - The building is situated in Grenoble, Isère department. Météo code: 074860.
- **Materials**
  - Outdoor Wall: Concrete and insulation
  - Window: Glass
  - Door: Wood
- **Height** $H = 3$ m
- **Surfaces**
  - $S_g = 1.2 \times 3 = 3.6\ m^2$ (Surface area of the glass)
  - $S_c = 26.7\ m^2$ (Surface area of the outdoor wall)
  - $S_b = 2.7\ m^2$ (Surface area of door)
- **Temperature control:** HVAC system

---


In [9]:
import numpy as np
import pandas as pd

# Geometry
H = 3.0
Sg = 3.6
Sc = 26.7
Sb = 2.7
Afloor = 30.0
Va = 90.0



**Thermo-Physical Properties**

| Material | Conductivity $\lambda$ [W/(m·K)] | Density $\rho$ [kg/m³] | Specific heat $c$ [J/(kg·K)] | Width $w$ [m] |
|----------|----------------------------------|------------------------|------------------------------|---------------|
| Air        | —     | 1.2   | 1000 | —    |
| Wood       | 0.2   | 310   | 1733 | 0.07 |
| Concrete   | 1.4   | 2300  | 880  | 0.32 |
| Insulation | 0.027 | 55    | 1210 | 0.08 |
| Glass      | 1.4   | 2500  | 1210 | 0.04 |

---

**Radiative Properties**

- $\varepsilon_{w,LW} = 0.85$ — Long wave emissivity: outdoor wall surface
- $\varepsilon_{g,LW} = 0.90$ — Long wave emissivity: glass
- $\alpha_{w,SW} = 0.25$ — Short wave absorptivity: white smooth surface
- $\alpha_{g,SW} = 0.38$ — Short wave absorptivity: reflective blue glass
- $\tau_{g,SW} = 0.30$ — Short wave transmittance: reflective blue glass

In [10]:
materials = {
    "concrete": {
        "rho": 2300,    # kg/m³  — density
        "c":   880,     # J/kg·K — specific heat
        "S":   26.7,    # m²     — surface area
        "w":   0.32,    # m      — thickness
    },
    "insulation": {
        "rho": 55,      # kg/m³
        "c":   1210,    # J/kg·K
        "S":   26.7,    # m²     — same surface as concrete wall
        "w":   0.08,    # m
    },
    "glass": {
        "rho": 2500,    # kg/m³
        "c":   1210,    # J/kg·K
        "S":   3.6,     # m²
        "w":   0.04,    # m
    },
    "wood": {
        "rho": 310,     # kg/m³
        "c":   1733,    # J/kg·K
        "S":   2.7,     # m²
        "w":   0.07,    # m
    },
    "air": {
        "rho": 1.2,     # kg/m³
        "c":   1000,    # J/kg·K
        "V":   90,      # m³     — room volume
    },
}

solar = {
    "epsilon_w_LW": 0.85,  # — long-wave emissivity: outdoor wall surface
    "epsilon_g_LW": 0.90,  # — long-wave emissivity: glass
    "alpha_w_SW": 0.25,    # — short-wave absorptivity: white smooth surface
    "alpha_g_SW": 0.38,    # — short-wave absorptivity: reflective blue glass
    "tau_g_SW": 0.30,      # — short-wave transmittance: reflective blue glass
}

## 1.2 Hypothesis

1. The temperature is uniform on each surface of the wall.
2. Heat transfer with the floor and ceiling is neglected (adiabatic).
3. Heat transfer with the indoor walls between apartments is neglected (adiabatic).
4. Heat transfers are assumed to be linear.
5. The indoor temperature is assumed to be homogeneous.

# 2. Thermal Circuit

<!-- Figure: thermalCircuit.png — Thermal Circuit -->

![Building Plan](thermalCircuit.png "Title")

# 3. Mathematical Model

**Summary**

There are 8 temperature nodes, numbered from $\theta_0$ to $\theta_7$; 14 heat-flow branches numbered from $q_0$ to $q_{13}$ with corresponding conductances $G_0$ to $G_{13}$; 1 temperature source $T_0$; 5 capacitances $C_0$ to $C_4$; 5 heat-flow sources $\dot{Q}_0$ to $\dot{Q}_3$ and $\dot{Q}_{aux}$.

## 3.1 Conductance Matrix G

**Wall Section**

| Conductance | Description | Formula | Calculation | Value (W/K) |
|-------------|-------------|---------|-------------|-------------|
| $G_0$ | Outdoor Convection     | $h_o S_c$            | $25 \times 26.7$                   | **667.50** |
| $G_1$ | Half Concrete Cond.    | $2\lambda_c S_c/w_c$ | $2 \times 1.4 \times 26.7/0.32$   | **233.63** |
| $G_2$ | Half Concrete Cond.    | $2\lambda_c S_c/w_c$ | $2 \times 1.4 \times 26.7/0.32$   | **233.63** |
| $G_3$ | Half Insulation Cond.  | $2\lambda_i S_c/w_i$ | $2 \times 0.027 \times 26.7/0.08$ | **18.02** |
| $G_4$ | Half Insulation Cond.  | $2\lambda_i S_c/w_i$ | $2 \times 0.027 \times 26.7/0.08$ | **18.02** |
| $G_5$ | Indoor Convection      | $h_i S_c$            | $8 \times 26.7$                    | **213.60** |

**Window Section**

| Conductance | Description | Formula | Calculation | Value (W/K) |
|-------------|-------------|---------|-------------|-------------|
| $G_8$ | Outdoor Convection | $h_o S_g$           | $25 \times 3.6$       | **90.00** |
| $G_7$ | Glass Conduction   | $\lambda_g S_g/w_g$ | $1.4 \times 3.6/0.04$ | **126.00** |
| $G_6$ | Indoor Convection  | $h_i S_g$           | $8 \times 3.6$        | **28.80** |

**Door Section**

| Conductance | Description | Formula | Calculation | Value (W/K) |
|-------------|-------------|---------|-------------|-------------|
| $G_9$    | Outdoor Convection     | $h_o S_b$            | $25 \times 2.7$                  | **67.50** |
| $G_{10}$ | Half Wood Conduction   | $2\lambda_b S_b/w_b$ | $2 \times 0.2 \times 2.7/0.07$  | **15.43** |
| $G_{11}$ | Half Wood Conduction   | $2\lambda_b S_b/w_b$ | $2 \times 0.2 \times 2.7/0.07$  | **15.43** |

**Ventilation and HVAC Section**

| Conductance | Description | Formula | Calculation | Value (W/K) |
|-------------|-------------|---------|-------------|-------------|
| $G_{12}$ | Advection       | $\rho c \dot{V}_a$ | $1.2 \times 1000 \times 0.025$ | **30.00** |
| $G_{13}$ | HVAC Controller | $K_p$              | —                              | — |

The conductance matrix $G$ is then given by:

$$
G = \begin{bmatrix}
G_0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & G_1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & G_2 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & G_3 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & G_4 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & G_5 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & G_6 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & G_7 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & G_8 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & G_9 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & G_{10} & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & G_{11} & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & G_{12} & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & G_{13}
\end{bmatrix}
$$

In [11]:
# Convection coefficients
ho = 25.0   # W/m²K
hi = 8.0    # W/m²K

# Thermal conductivity values used in the report
lam_concrete = 1.4
lam_insulation = 0.027
lam_glass = 1.4
lam_wood = 0.2

# Surface areas
Sc = materials["concrete"]["S"]
Si = materials["insulation"]["S"]
Sg = materials["glass"]["S"]
Sb = materials["wood"]["S"]

# Thicknesses
wc = materials["concrete"]["w"]
wi = materials["insulation"]["w"]
wg = materials["glass"]["w"]
wb = materials["wood"]["w"]

# Air properties
rho_air = materials["air"]["rho"]
c_air = materials["air"]["c"]
Va_dot = 0.025  # m³/s

# Wall section
G0 = ho * Sc
G1 = 2 * lam_concrete * Sc / wc
G2 = 2 * lam_concrete * Sc / wc
G3 = 2 * lam_insulation * Sc / wi
G4 = 2 * lam_insulation * Sc / wi
G5 = hi * Sc

# Window section
G8 = ho * Sg
G7 = lam_glass * Sg / wg
G6 = hi * Sg

# Door section
G9 = ho * Sb
G10 = 2 * lam_wood * Sb / wb
G11 = 2 * lam_wood * Sb / wb

# Ventilation / HVAC
G12 = rho_air * c_air * Va_dot
G13 = None  # HVAC controller gain 

conductances = {
    "G0": G0,
    "G1": G1,
    "G2": G2,
    "G3": G3,
    "G4": G4,
    "G5": G5,
    "G6": G6,
    "G7": G7,
    "G8": G8,
    "G9": G9,
    "G10": G10,
    "G11": G11,
    "G12": G12,
    "G13": G13,
}

print(conductances)

{'G0': 667.5, 'G1': 233.62499999999997, 'G2': 233.62499999999997, 'G3': 18.0225, 'G4': 18.0225, 'G5': 213.6, 'G6': 28.8, 'G7': 126.0, 'G8': 90.0, 'G9': 67.5, 'G10': 15.428571428571429, 'G11': 15.428571428571429, 'G12': 30.0, 'G13': None}


## 3.2 Incidence Matrix A

An element $A_{k,j}$ is 1 if flow $q_k$ enters node $\theta_j$, -1 if it leaves, and 0 otherwise. Therefore, based on the thermal circuit, the incidence matrix $A$ is given as below.

$$
A = \begin{bmatrix}
 1  &  0  &  0  &  0  &  0  &  0  &  0  &  0  &  0  &  0 \\
-1  &  1  &  0  &  0  &  0  &  0  &  0  &  0  &  0  &  0 \\
 0  & -1  &  1  &  0  &  0  &  0  &  0  &  0  &  0  &  0 \\
 0  &  0  & -1  &  1  &  0  &  0  &  0  &  0  &  0  &  0 \\
 0  &  0  &  0  & -1  &  1  &  0  &  0  &  0  &  0  &  0 \\
 0  &  0  &  0  &  0  & -1  &  1  &  0  &  0  &  0  &  0 \\
 0  &  0  &  0  &  0  &  0  &  1  & -1  &  0  &  0  &  0 \\
 0  &  0  &  0  &  0  &  0  &  0  &  1  & -1  &  0  &  0 \\
 0  &  0  &  0  &  0  &  0  &  0  &  0  &  1  &  0  &  0 \\
 0  &  0  &  0  &  0  &  0  &  0  &  0  &  0  &  1  &  0 \\
 0  &  0  &  0  &  0  &  0  &  0  &  0  &  0  & -1  &  1 \\
 0  &  0  &  0  &  0  &  0  &  1  &  0  &  0  &  0  & -1 \\
 0  &  0  &  0  &  0  &  0  &  1  &  0  &  0  &  0  &  0 \\
 0  &  0  &  0  &  0  &  0  &  1  &  0  &  0  &  0  &  0
\end{bmatrix}
$$

In [12]:

A = np.array([
    [ 1,  0,  0,  0,  0,  0,  0,  0,  0,  0],
    [-1,  1,  0,  0,  0,  0,  0,  0,  0,  0],
    [ 0, -1,  1,  0,  0,  0,  0,  0,  0,  0],
    [ 0,  0, -1,  1,  0,  0,  0,  0,  0,  0],
    [ 0,  0,  0, -1,  1,  0,  0,  0,  0,  0],
    [ 0,  0,  0,  0, -1,  1,  0,  0,  0,  0],
    [ 0,  0,  0,  0,  0,  1, -1,  0,  0,  0],
    [ 0,  0,  0,  0,  0,  0,  1, -1,  0,  0],
    [ 0,  0,  0,  0,  0,  0,  0,  1,  0,  0],
    [ 0,  0,  0,  0,  0,  0,  0,  0,  1,  0],
    [ 0,  0,  0,  0,  0,  0,  0,  0, -1,  1],
    [ 0,  0,  0,  0,  0,  1,  0,  0,  0, -1],
    [ 0,  0,  0,  0,  0,  1,  0,  0,  0,  0],
    [ 0,  0,  0,  0,  0,  1,  0,  0,  0,  0],
], dtype=float)

print(A)

[[ 1.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [-1.  1.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0. -1.  1.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. -1.  1.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0. -1.  1.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0. -1.  1.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  1. -1.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  1. -1.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  1.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  1.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0. -1.  1.]
 [ 0.  0.  0.  0.  0.  1.  0.  0.  0. -1.]
 [ 0.  0.  0.  0.  0.  1.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  1.  0.  0.  0.  0.]]


## 3.3 Capacity Matrix C

From the room plan, the following capacities are modelled:

- $C_0$ at node $\theta_1$: Represents the thermal mass of the concrete layer in the wall.
- $C_1$ at node $\theta_3$: Represents the thermal mass of the insulation layer in the wall.
- $C_2$ at node $\theta_6$: Represents the thermal mass of the glass window.
- $C_3$ at node $\theta_7$: Represents the thermal mass of the wooden door.
- $C_4$ at node $\theta_5$: Represents the thermal mass of the indoor air volume.

The $C$ matrix is a diagonal matrix. If the node has a capacity attached to it, the value is assigned.

| Capacity | Node | Component | Formula | Calculation | Value (J/K) |
|----------|------|-----------|---------|-------------|-------------|
| $C_0$ | $\theta_1$ | Concrete Layer   | $\rho_c c_c S_c w_c$ | $2300 \cdot 880 \cdot 26.7 \cdot 0.32$  | **17,293,120** |
| $C_1$ | $\theta_3$ | Insulation Layer | $\rho_i c_i S_c w_i$ | $55 \cdot 1210 \cdot 26.7 \cdot 0.08$   | **142,152** |
| $C_2$ | $\theta_7$ | Glass Window     | $\rho_g c_g S_g w_g$ | $2500 \cdot 1210 \cdot 3.6 \cdot 0.04$  | **435,600** |
| $C_3$ | $\theta_8$ | Wooden Door      | $\rho_b c_b S_b w_b$ | $310 \cdot 1733 \cdot 2.7 \cdot 0.07$   | **101,507** |
| $C_4$ | $\theta_5$ | Indoor Air       | $\rho_a c_a V_a$     | $1.2 \cdot 1000 \cdot 90$               | **108,000** |

$$
C = \begin{bmatrix}
0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   \\
0   & C_0 & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   \\
0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   \\
0   & 0   & 0   & C_1 & 0   & 0   & 0   & 0   & 0   & 0   \\
0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   \\
0   & 0   & 0   & 0   & 0   & C_4 & 0   & 0   & 0   & 0   \\
0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   \\
0   & 0   & 0   & 0   & 0   & 0   & 0   & C_2 & 0   & 0   \\
0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & C_3 & 0   \\
0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0   & 0
\end{bmatrix}
$$

In [13]:

# Capacitances
C0 = materials["concrete"]["rho"] * materials["concrete"]["c"] * materials["concrete"]["S"] * materials["concrete"]["w"]
C1 = materials["insulation"]["rho"] * materials["insulation"]["c"] * materials["insulation"]["S"] * materials["insulation"]["w"]
C2 = materials["glass"]["rho"] * materials["glass"]["c"] * materials["glass"]["S"] * materials["glass"]["w"]
C3 = materials["wood"]["rho"] * materials["wood"]["c"] * materials["wood"]["S"] * materials["wood"]["w"]
C4 = materials["air"]["rho"] * materials["air"]["c"] * materials["air"]["V"]

# Matrix C
C = np.diag([0, C0, 0, C1, 0, C4, 0, C2, C3, 0])

print(C)

[[       0.          0.          0.          0.          0.          0.
         0.          0.          0.          0.  ]
 [       0.   17293056.          0.          0.          0.          0.
         0.          0.          0.          0.  ]
 [       0.          0.          0.          0.          0.          0.
         0.          0.          0.          0.  ]
 [       0.          0.          0.     142150.8         0.          0.
         0.          0.          0.          0.  ]
 [       0.          0.          0.          0.          0.          0.
         0.          0.          0.          0.  ]
 [       0.          0.          0.          0.          0.     108000.
         0.          0.          0.          0.  ]
 [       0.          0.          0.          0.          0.          0.
         0.          0.          0.          0.  ]
 [       0.          0.          0.          0.          0.          0.
         0.     435600.          0.          0.  ]
 [       0.     

## 3.4 Temperature Source Vector b

The temperature source vector assigns a value when the temperature source is attached to the branch of heat flow. Therefore, the vector is:

$$
b = [T_0,\, 0,\, 0,\, 0,\, 0,\, 0,\, 0,\, 0,\, T_0,\, T_0,\, 0,\, 0,\, T_0,\, T_i]^T
$$

In [14]:
T0 = To
b = np.array([T0, 0, 0, 0, 0, 0, 0, 0, T0, T0, 0, 0, T0, Ti], dtype=float)

print(b)

NameError: name 'To' is not defined

## 3.5 Heat Flow Source Vector f

The heat flow source vector assigns a value when the heat flow source is attached to the branch of heat flow.

| Source | Formula | Calculation | Value (W) |
|--------|---------|-------------|----------|
| $\dot{Q}_0$     | $\alpha_{w,SW} S_c E$             | $0.25 \cdot 26.7 \cdot 200$           | **667.50** |
| $\dot{Q}_1$     | $\tau_{g,SW} \alpha_{w,SW} S_g E$ | $0.30 \cdot 0.25 \cdot 3.6 \cdot 200$ | **54.00** |
| $\dot{Q}_2$     | $\alpha_{g,SW} S_g E$             | $0.38 \cdot 3.6 \cdot 200$            | **273.60** |
| $\dot{Q}_3$     | $\alpha_{w,SW} S_b E$             | $0.25 \cdot 2.7 \cdot 200$            | **135.00** |
| $\dot{Q}_{aux}$ | —                                 | Assumed (e.g., 0 or 1000)             | **0.00** |

$$
f = [\dot{Q}_0,\, 0,\, 0,\, 0,\, \dot{Q}_1,\, \dot{Q}_{aux},\, 0,\, \dot{Q}_2,\, \dot{Q}_3,\, 0]^T
$$

In [ ]:
Q0 = solar["alpha_w_SW"] * materials["concrete"]["S"] * solar["E"]
Q1 = solar["tau_g_SW"] * solar["alpha_w_SW"] * materials["glass"]["S"] * solar["E"]
Q2 = solar["alpha_g_SW"] * materials["glass"]["S"] * solar["E"]
Q3 = solar["alpha_w_SW"] * materials["wood"]["S"] * solar["E"]
Q_aux = 0.0

f = np.array([Q0, 0, 0, 0, Q1, Q_aux, 0, Q2, Q3, 0], dtype=float)

print(f)

[1335.     0.     0.     0.    54.     0.     0.   273.6  135.     0. ]


# 4. Thermal Load and Yearly Consumption

In steady-state (all time derivatives vanish, $\dot{\theta} = 0$), the DAE system reduces to a purely algebraic system. For a **perfect HVAC controller** ($K_p \rightarrow \infty$), the indoor temperature $\theta_5$ is forced to the setpoint $T_i$. The HVAC load then equals the total net heat exchange with the exterior.

The **overall thermal conductance** between the outdoor air and the indoor air is computed by combining, in series, the conductances along each heat-transfer path:

$$
UA_\text{wall} = \frac{1}{\frac{1}{G_0} + \frac{1}{G_1} + \frac{1}{G_2} + \frac{1}{G_3} + \frac{1}{G_4} + \frac{1}{G_5}} = \frac{1}{\frac{1}{667.50} + \frac{2}{233.63} + \frac{2}{18.02} + \frac{1}{213.60}} = \mathbf{7.95\ \text{W/K}}
$$

$$
UA_\text{win} = \frac{1}{\frac{1}{G_8} + \frac{1}{G_7} + \frac{1}{G_6}} = \frac{1}{\frac{1}{90.00} + \frac{1}{126.00} + \frac{1}{28.80}} = \mathbf{18.60\ \text{W/K}}
$$

$$
UA_\text{door} = \frac{1}{\frac{1}{G_9} + \frac{1}{G_{10}} + \frac{1}{G_{11}}} = \frac{1}{\frac{1}{67.50} + \frac{2}{15.43}} = \mathbf{6.92\ \text{W/K}}
$$

$$
H_\text{total} = UA_\text{wall} + UA_\text{win} + UA_\text{door} + G_{12} = 7.95 + 18.60 + 6.92 + 30.00 = \mathbf{63.47\ \text{W/K}}
$$

| Element | Series path | $UA$ [W/K] | $U$ [W/(m²·K)] | Surface [m²] |
|---------|-------------|-----------|----------------|-------------|
| Outdoor wall | $G_0 \to G_1 \to G_2 \to G_3 \to G_4 \to G_5$ | 7.95 | 0.298 | 26.7 |
| Window | $G_8 \to G_7 \to G_6$ | 18.60 | 5.17 | 3.6 |
| Door | $G_9 \to G_{10} \to G_{11}$ | 6.92 | 2.56 | 2.7 |
| Ventilation | $G_{12}$ (advection) | 30.00 | — | — |
| **Total** $H_\text{total}$ | All paths | **63.47** | — | — |

In [16]:
UAwall = 1.0 / (1/G0 + 1/G1 + 1/G2 + 1/G3 + 1/G4 + 1/G5)
UAwin = 1.0 / (1/G8 + 1/G7 + 1/G6)
UAdoor = 1.0 / (1/G9 + 1/G10 + 1/G11)
Htotal = UAwall + UAwin + UAdoor + G12

print(f"UAwall = {UAwall:.2f} W/K")
print(f"UAwin  = {UAwin:.2f} W/K")
print(f"UAdoor = {UAdoor:.2f} W/K")
print(f"Htotal = {Htotal:.2f} W/K")


UAwall = 7.95 W/K
UAwin  = 18.60 W/K
UAdoor = 6.92 W/K
Htotal = 63.48 W/K


## 4.1 Thermal Load

### 4.1.1 Most Unfavorable Condition

The most unfavorable conditions are defined below, together with the resulting HVAC load.

#### Maximum Heating Load (Winter)

- Lowest outdoor temperature: $T_o = -10\ °\text{C}$ (Grenoble design winter temperature, RT2012/RE2020)
- Indoor setpoint: $T_{i,sp} = 20\ °\text{C}$
- Cloudy sky: no solar radiation ($E = 0\ \text{W/m}^2$)
- No internal heat sources ($\dot{Q}_\text{aux} = 0\ \text{W}$) — worst-case night scenario
- Ventilation active: $\text{ACH} = 1\ \text{h}^{-1}$ ($\dot{V}_a = 0.025\ \text{m}^3/\text{s}$, $G_{12} = 30\ \text{W/K}$)

$$\Delta T_\text{win} = T_{i,sp} - T_o = 20 - (-10) = 30\ \text{K}$$

| Heat Loss Path | Formula | Value [W] |
|----------------|---------|----------|
| Outdoor wall  | $UA_\text{wall} \times \Delta T = 7.95 \times 30$  | 238.5  |
| Window        | $UA_\text{win}  \times \Delta T = 18.60 \times 30$ | 558.0  |
| Door          | $UA_\text{door} \times \Delta T = 6.92 \times 30$  | 207.6  |
| Ventilation   | $G_{12} \times \Delta T = 30.00 \times 30$         | 900.0  |
| **HVAC heating load** $\dot{Q}_\text{heat}$ | $H_\text{total} \times \Delta T = 63.47 \times 30$ | **1904.1** |

$$\boxed{\dot{Q}_\text{heat} = H_\text{total} \times \Delta T_\text{win} = 63.47 \times 30 \approx \mathbf{1904\ \text{W}} \approx 1.90\ \text{kW}}$$

Specific heating power: $\dot{Q}_\text{heat} / A_\text{floor} = 1904 / 30 \approx 63.5\ \text{W/m}^2$.

#### Maximum Cooling Load (Summer)

- Highest outdoor temperature: $T_o = 35\ °\text{C}$ (Grenoble peak summer, RT2012/RE2020)
- Indoor setpoint: $T_{i,sp} = 26\ °\text{C}$
- Peak solar irradiance: $E = 500\ \text{W/m}^2$ on south-facing surfaces
- No internal heat sources ($\dot{Q}_\text{aux} = 0\ \text{W}$)

$$\Delta T_\text{sum} = T_o - T_{i,sp} = 35 - 26 = 9\ \text{K}$$

The total cooling load includes the envelope transmission, solar radiation transmitted through the window, and solar gains conducted through the opaque surfaces:

$$\dot{Q}_\text{env} = H_\text{total} \times \Delta T_\text{sum} = 63.47 \times 9 = 571.2\ \text{W}$$

$$\dot{\Phi}_\text{win} = \tau_{g,SW} \cdot S_g \cdot E = 0.30 \times 3.6 \times 500 = 540.0\ \text{W}$$

$$\dot{\Phi}_\text{wall} = \alpha_{w,SW} \cdot S_c \cdot E \cdot \frac{UA_\text{wall}}{G_0} = 0.25 \times 26.7 \times 500 \times \frac{7.95}{667.50} = 39.8\ \text{W}$$

$$\dot{\Phi}_\text{door} = \alpha_{w,SW} \cdot S_b \cdot E \cdot \frac{UA_\text{door}}{G_9} = 0.25 \times 2.7 \times 500 \times \frac{6.92}{67.50} = 34.6\ \text{W}$$

| Cooling Load Component | Formula | Value [W] |
|------------------------|---------|----------|
| Envelope transmission        | $H_\text{total} \times \Delta T = 63.47 \times 9$             | 571.2  |
| Solar gain through window    | $\tau_{g,SW} \cdot S_g \cdot E = 0.30 \times 3.6 \times 500$ | 540.0  |
| Solar gain via wall (cond.)  | $\alpha_{w,SW} \cdot S_c \cdot E \cdot (UA_\text{wall}/G_0)$ | 39.8   |
| Solar gain via door (cond.)  | $\alpha_{w,SW} \cdot S_b \cdot E \cdot (UA_\text{door}/G_9)$ | 34.6   |
| **HVAC cooling load** $\dot{Q}_\text{cool}$ | Sum of above | **1185.6** |

$$\boxed{\dot{Q}_\text{cool} = 1185.6\ \text{W} \approx 1.19\ \text{kW}}$$

#### Conclusion

| Condition | $T_o$ [°C] | $T_{i,sp}$ [°C] | $\Delta T$ [K] | $E$ [W/m²] | HVAC Load [W] |
|-----------|-----------|----------------|--------------|-----------|---------------|
| Winter (heating) | $-10$ | $20$ | $30$ | $0$   | **1904** |
| Summer (cooling) | $35$  | $26$ | $9$  | $500$ | 1186 |

The **winter heating condition is the most unfavorable**, producing the higher HVAC demand. The summer cooling load is dominated by solar gains through the south-facing window ($\dot{\Phi}_\text{win} = 540\ \text{W}$, representing 45.6% of the total cooling load).

In [17]:
# -----------------------------
# Worst-case winter
# -----------------------------
To_winter = -10.0
Ti_winter = 20.0
dT_winter = Ti_winter - To_winter

Q_wall_heat = UAwall * dT_winter
Q_win_heat = UAwin * dT_winter
Q_door_heat = UAdoor * dT_winter
Q_vent_heat = G12 * dT_winter
Q_heat = Htotal * dT_winter

# -----------------------------
# Worst-case summer
# -----------------------------
To_summer = 35.0
Ti_summer = 26.0
E_summer = 500.0
dT_summer = To_summer - Ti_summer

Q_env = Htotal * dT_summer
Phi_win = solar["tau_g_SW"] * materials["glass"]["S"] * E_summer
Phi_wall = solar["alpha_w_SW"] * materials["concrete"]["S"] * E_summer * (UAwall / G0)
Phi_door = solar["alpha_w_SW"] * materials["wood"]["S"] * E_summer * (UAdoor / G9)
Q_cool = Q_env + Phi_win + Phi_wall + Phi_door

# -----------------------------
# Summary table
# -----------------------------
thermal_load_summary = pd.DataFrame([
    ["Winter (heating)", To_winter, Ti_winter, dT_winter, 0.0, Q_heat],
    ["Summer (cooling)", To_summer, Ti_summer, dT_summer, E_summer, Q_cool],
], columns=["Condition", "To (°C)", "Ti,sp (°C)", "ΔT (K)", "E (W/m²)", "HVAC load (W)"])

print("Heating load breakdown")
print(f"Outdoor wall: {Q_wall_heat:.1f} W")
print(f"Window:       {Q_win_heat:.1f} W")
print(f"Door:         {Q_door_heat:.1f} W")
print(f"Ventilation:  {Q_vent_heat:.1f} W")
print(f"Total heating load: {Q_heat:.1f} W")

print("\nCooling load breakdown")
print(f"Envelope transmission: {Q_env:.1f} W")
print(f"Window solar gain:     {Phi_win:.1f} W")
print(f"Wall solar gain:       {Phi_wall:.1f} W")
print(f"Door solar gain:       {Phi_door:.1f} W")
print(f"Total cooling load:    {Q_cool:.1f} W")

print("\nSummary")
print(thermal_load_summary.to_string(index=False))

Heating load breakdown
Outdoor wall: 238.6 W
Window:       557.9 W
Door:         207.7 W
Ventilation:  900.0 W
Total heating load: 1904.3 W

Cooling load breakdown
Envelope transmission: 571.3 W
Window solar gain:     540.0 W
Wall solar gain:       39.8 W
Door solar gain:       34.6 W
Total cooling load:    1185.7 W

Summary
       Condition  To (°C)  Ti,sp (°C)  ΔT (K)  E (W/m²)  HVAC load (W)
Winter (heating)    -10.0        20.0    30.0       0.0    1904.264884
Summer (cooling)     35.0        26.0     9.0     500.0    1185.668016


## 4.2 Yearly Energy Consumption

Annual energy consumption is estimated using the **degree-day method**, which integrates the thermal balance over the year using Heating Degree Days (HDD) and Cooling Degree Days (CDD).

**Climate Data — Grenoble (Météo France, code 074860)**

| Parameter | Value | Unit | Notes |
|-----------|-------|------|-------|
| Heating Degree Days (base 20°C) | 3 260 | °C·day/year | Continental alpine climate |
| Cooling Degree Days (base 26°C) | 150   | °C·day/year | Mild summers in alpine valley |
| Solar irradiation — south vertical | 1 200 | kWh/(m²·year) | Annual total, south façade |
| Heating season duration | ≈ 212 | days/year | October – April |
| Cooling season duration | ≈ 90  | days/year | June – August |

**Heating Energy**

The gross annual heating demand is computed from the degree-day formula:

$$E_{\text{heat,gross}} = H_\text{total} \times \text{HDD} \times 24\ \frac{\text{h}}{\text{day}} = 63.47 \times 3260 \times 24 = 4{,}967{,}443\ \text{Wh/year} \approx \mathbf{4\,967\ \text{kWh/year}}$$

Useful solar gains through the south-facing window reduce the net demand. A solar utilization factor $\eta_\text{sol} = 0.50$ accounts for the fact that gains are not always available when heating is needed:

$$E_\text{solar} = \tau_{g,SW} \cdot S_g \cdot H_\text{south} \cdot \eta_\text{sol} = 0.30 \times 3.6 \times 1200 \times 0.50 = \mathbf{648\ \text{kWh/year}}$$

Net heating demand and delivered energy, assuming HVAC system efficiency $\eta_\text{HVAC} = 0.85$:

$$E_{\text{heat,net}} = E_{\text{heat,gross}} - E_\text{solar} = 4967 - 648 = \mathbf{4\,319\ \text{kWh/year}}$$

$$E_{\text{heat,delivered}} = \frac{E_{\text{heat,net}}}{\eta_\text{HVAC}} = \frac{4319}{0.85} = \mathbf{5\,081\ \text{kWh/year}}$$

**Cooling Energy**|

Cooling energy is estimated from the CDD method plus solar gains through the window during summer, using a coefficient of performance $\text{COP} = 3.5$:

$$E_{\text{cool,env}} = \frac{H_\text{total} \times \text{CDD} \times 24}{\text{COP} \times 1000} = \frac{63.47 \times 150 \times 24}{3.5 \times 1000} = \mathbf{65.3\ \text{kWh/year}}$$

$$E_{\text{cool,solar}} = \frac{\tau_{g,SW} \cdot S_g \cdot H_\text{summer}}{\text{COP}} = \frac{0.30 \times 3.6 \times 300}{3.5} = \mathbf{92.6\ \text{kWh/year}}$$

$$E_{\text{cool,delivered}} = E_{\text{cool,env}} + E_{\text{cool,solar}} = \mathbf{157.9\ \text{kWh/year}}$$

**Annual Energy Summary**

| Energy Component | Value [kWh/year] | Share | Comment |
|-----------------|-----------------|-------|--------|
| Gross heating demand         |  4 967 | —    | Degree-day method, $H = 63.47\ \text{W/K}$ |
| Solar gains offset (heating) | −648   | —    | South window, $\eta_\text{sol} = 0.50$ |
| Net heating demand           |  4 319 | —    | After solar offset |
| Heating energy delivered     |  5 081 | 97%  | $\eta_\text{HVAC} = 0.85$ |
| Cooling energy delivered     |    158 |  3%  | $\text{COP} = 3.5$ |
| **Total** $E_\text{total}$   | **5 239** | **100%** | |
| Floor area $A_\text{floor}$  | $30\ \text{m}^2$ | — | |
| **Specific consumption** $e$ | **175 kWh/(m²·year)** | — | RT2012 ref: 50–80 kWh/(m²·yr) |

The specific energy consumption of 175kWh/$m^2$.year is dominated by heating (97% of total), consistent with Grenoble's continental alpine climate. The main driver is ventilation loss: $G_{12} = 30\ \text{W/K}$ contributes 47% of $H_\text{total}$. Installing mechanical heat recovery ventilation (MVHR) with 80% efficiency would reduce the effective ventilation conductance to approximately $6\ \text{W/K}$, saving roughly $1\,500\ \text{kWh/year}$ in heating energy alone.

In [18]:
# Annual energy consumption

HDD = 3260        # °C·day/year
CDD = 150         # °C·day/year
Hsouth = 1200     # kWh/m²·year
Hsummer = 300     # kWh/m²·year
eta_sol = 0.50
eta_HVAC = 0.85
COP = 3.5

# Heating
Eheat_gross = Htotal * HDD * 24 / 1000              # kWh/year
Esolar = solar["tau_g_SW"] * materials["glass"]["S"] * Hsouth * eta_sol
Eheat_net = Eheat_gross - Esolar
Eheat_delivered = Eheat_net / eta_HVAC

# Cooling
Ecool_env = Htotal * CDD * 24 / (COP * 1000)        # kWh/year
Ecool_solar = solar["tau_g_SW"] * materials["glass"]["S"] * Hsummer / COP
Ecool_delivered = Ecool_env + Ecool_solar

# Total
Etotal = Eheat_delivered + Ecool_delivered
Afloor = 30.0
e_specific = Etotal / Afloor

print(f"Eheat_gross      = {Eheat_gross:.1f} kWh/year")
print(f"Esolar           = {Esolar:.1f} kWh/year")
print(f"Eheat_net        = {Eheat_net:.1f} kWh/year")
print(f"Eheat_delivered  = {Eheat_delivered:.1f} kWh/year")
print(f"Ecool_delivered  = {Ecool_delivered:.1f} kWh/year")
print(f"Etotal           = {Etotal:.1f} kWh/year")
print(f"e_specific       = {e_specific:.1f} kWh/(m²·year)")

Eheat_gross      = 4966.3 kWh/year
Esolar           = 648.0 kWh/year
Eheat_net        = 4318.3 kWh/year
Eheat_delivered  = 5080.4 kWh/year
Ecool_delivered  = 157.9 kWh/year
Etotal           = 5238.2 kWh/year
e_specific       = 174.6 kWh/(m²·year)
